# Copilot Licensed Users — Direct Ingester (Fabric)

End-to-end Lakehouse loader for **Copilot licensed users** that calls Microsoft Graph directly from inside Fabric. Replaces the prior flow:

```
OLD:  PowerShell → CSV → Files/licensed_raw/ → Copilot_Licensed_Users_Loader.ipynb → Delta
NEW:  This notebook (Graph → Delta)
```

**Source endpoint**: `/v1.0/reports/getOffice365ActiveUserDetail(period='D7')` — returns the M365 active-user report. Each row is enriched with `HasCopilot = TRUE` if the user has Microsoft 365 Copilot in their assigned products list.

**Output**: Lakehouse Delta table `dbo.copilot_licensed_users` (same name + schema as the existing CSV loader, so the PBIT works without changes).

**Permissions**: app registration with `Reports.Read.All` (Application permission, admin-consented).

**Heads-up — masked UPNs**: if the report shows 32-char hex strings instead of real UPNs, an admin needs to untick **Org settings → Reports → "Display concealed user, group, and site names in all reports"** in `admin.microsoft.com`.


## 1. Configuration

Same app reg + secret as the audit-log direct ingester. The output table is schema-qualified (`dbo.`) to work with both schema-enabled and legacy Lakehouses.


In [ ]:
# === CONFIG ===
TENANT_ID     = '<your-tenant-guid>'
CLIENT_ID     = '<your-app-reg-client-id>'
CLIENT_SECRET = '<your-client-secret-value>'

REPORT_PERIOD = 'D7'                                # 'D7', 'D30', 'D90', 'D180' — Microsoft fixed values
OUTPUT_TABLE  = 'dbo.copilot_licensed_users'        # Delta table consumed by the PBIT
WRITE_MODE    = 'overwrite'                         # 'overwrite' for full snapshots


## 2. Authenticate to Microsoft Graph

In [ ]:
import requests

def get_graph_token(tenant_id, client_id, client_secret):
    url  = f'https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token'
    body = {
        'client_id':     client_id,
        'scope':         'https://graph.microsoft.com/.default',
        'client_secret': client_secret,
        'grant_type':    'client_credentials',
    }
    r = requests.post(url, data=body, timeout=30)
    r.raise_for_status()
    return r.json()['access_token']

token   = get_graph_token(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
headers = {'Authorization': f'Bearer {token}'}
print('✓ Graph token acquired.')


## 3. Fetch the M365 active user report

Graph's `getOffice365ActiveUserDetail` endpoint returns a **CSV** body (not JSON). We parse it into a Spark DataFrame.


In [ ]:
import csv
from io import StringIO

report_uri = f"https://graph.microsoft.com/v1.0/reports/getOffice365ActiveUserDetail(period='{REPORT_PERIOD}')"
r = requests.get(report_uri, headers=headers, timeout=120)
r.raise_for_status()

# Strip UTF-8 BOM if present, parse as CSV
csv_text = r.text.lstrip('\ufeff')
reader = csv.DictReader(StringIO(csv_text))
rows = list(reader)
print(f'✓ Fetched {len(rows):,} rows from {report_uri}')
if rows:
    print('Sample columns:', list(rows[0].keys())[:8], '...')


## 4. Derive `HasCopilot` flag

Microsoft's report has an `Assigned Products` column listing all SKUs assigned to the user. We flag any row containing `MICROSOFT 365 COPILOT` as having a Copilot license.


In [ ]:
for row in rows:
    products = (row.get('Assigned Products') or '').upper()
    row['HasCopilot'] = 'TRUE' if 'MICROSOFT 365 COPILOT' in products else 'FALSE'

copilot_count = sum(1 for r in rows if r['HasCopilot'] == 'TRUE')
print(f'  Total users:          {len(rows):,}')
print(f'  With Copilot license: {copilot_count:,}')


## 5. Build Spark DataFrame + canonicalise columns

Mirrors the existing CSV loader's column-renaming logic. UPN variants → `User Principal Name`, then sanitise spaces in column names (Delta forbids ` ,;{}()\n\t=`).


In [ ]:
import re
from pyspark.sql import functions as F

raw = spark.createDataFrame(rows)

# Detect UPN column variants (Graph returns 'User Principal Name' but be defensive)
upn_variants = ['User Principal Name', 'userPrincipalName', 'UserPrincipalName', 'User principal name']
actual_upn = next((c for c in upn_variants if c in raw.columns), None)
df = raw
if actual_upn and actual_upn != 'User Principal Name':
    df = df.withColumnRenamed(actual_upn, 'User Principal Name')

# Add canonical 'Has license' column from HasCopilot (alias to match existing loader output)
df = df.withColumnRenamed('HasCopilot', 'Has license')

# Add normalized join key + dedupe on it
df = df.withColumn(
    'UPN_Normalized',
    F.when(F.col('`User Principal Name`').isNull(), None)
     .otherwise(F.lower(F.trim(F.col('`User Principal Name`').cast('string'))))
)
df = df.dropDuplicates(['UPN_Normalized'])

# Sanitize column names (Delta forbids spaces / punctuation)
_INVALID = re.compile(r'[ ,;{}()\n\t=]')
df = df.toDF(*[_INVALID.sub('_', c) for c in df.columns])

print(f'After dedupe + sanitize: {df.count():,} rows')
print('Columns:', df.columns)


## 6. Write to Lakehouse Delta table

In [ ]:
(df.write
    .format('delta')
    .mode(WRITE_MODE)
    .option('overwriteSchema', 'true')
    .saveAsTable(OUTPUT_TABLE))

row_count = spark.table(OUTPUT_TABLE).count()
print(f'✓ Rows written to {OUTPUT_TABLE}: {row_count:,}')


## 7. Verify

In [ ]:
tbl = spark.table(OUTPUT_TABLE)
tbl.select('User_Principal_Name', 'Has_license', 'UPN_Normalized').show(10, truncate=False)
tbl.groupBy('Has_license').count().orderBy(F.desc('count')).show()


---
**Connect the PBIT**: this table is consumed by the `Copilot Licensed` query in both AI-in-One and AI Business Value dashboards. Once this notebook has run, leave the `Copilot Licensed Users` parameter blank when opening the PBIT — refresh sources from `dbo.copilot_licensed_users` directly via the Fabric SQL endpoint.
